<div style="background:linear-gradient(135deg,#0F3D6E 0%,#2176AE 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#C8DEF5;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 1 · CLASE 3</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: Gauss-Markov, BLUE y Diagnósticos</h1>
  <p style="color:#C8DEF5;font-size:16px;margin:0 0 24px 0;font-style:italic;">Verificar los supuestos antes de interpretar cualquier resultado</p>
  <hr style="border-color:#5BA4CF;margin:0 0 20px 0;">
  <p style="color:#EAF2FB;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,
                     'axes.spines.top':False,'axes.spines.right':False})
SEED = 42; np.random.seed(SEED)
print('✅ Setup listo')

---
## 🟢 Ejercicio 1 — Verificar s² = SSE/(n−p) es insesgado

Dados los tres estimadores de σ² abajo:
1. Simular 4000 datasets del mismo modelo
2. Calcular cada estimador en cada simulación
3. Comparar E[estimador] con σ² real
4. ¿Cuál es insesgado? ¿Por qué?
5. Graficar la distribución de los tres estimadores

In [ ]:
np.random.seed(SEED)
n_obs, p_params = 50, 4
beta_v  = np.array([1.0, 2.0, -1.0, 0.5])
sigma2_v = 9.0   # σ² real

X_v = np.column_stack([np.ones(n_obs), np.random.randn(n_obs, p_params-1)])

# Tres estimadores:
# A) SSE / n       — sesgado
# B) SSE / (n-1)   — parcialmente corregido
# C) SSE / (n-p)   — insesgado (correcto)

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n_obs, p_params = 50, 4
beta_v = np.array([1.,2.,-1.,0.5]); sigma2_v = 9.
X_v    = np.column_stack([np.ones(n_obs), np.random.randn(n_obs, p_params-1)])
n_sims = 4000
ests   = {'SSE/n':[],'SSE/(n-1)':[],'SSE/(n-p)':[]}

for _ in range(n_sims):
    eps = np.random.normal(0, np.sqrt(sigma2_v), n_obs)
    y_s = X_v @ beta_v + eps
    b_s, *_ = np.linalg.lstsq(X_v, y_s, rcond=None)
    SSE_s = ((y_s - X_v @ b_s)**2).sum()
    ests['SSE/n'].append(SSE_s / n_obs)
    ests['SSE/(n-1)'].append(SSE_s / (n_obs-1))
    ests['SSE/(n-p)'].append(SSE_s / (n_obs-p_params))

print(f'σ² real = {sigma2_v}')
for nm, vals in ests.items():
    arr = np.array(vals)
    sesgo = arr.mean() - sigma2_v
    insesg = '✓ insesgado' if abs(sesgo) < 0.1 else '✗ sesgado'
    print(f'  E[{nm:12s}] = {arr.mean():.3f}  sesgo={sesgo:+.3f}  {insesg}')

fig, axes = plt.subplots(1,3,figsize=(13,4))
for ax,(nm,vals) in zip(axes,ests.items()):
    ax.hist(vals, bins=50, density=True, color='#2176AE', edgecolor='white', alpha=0.8)
    ax.axvline(sigma2_v, color='#C0392B', lw=2, label=f'σ²={sigma2_v}')
    ax.axvline(np.mean(vals), color='#0F3D6E', lw=1.5, linestyle='--',
               label=f'E[est]={np.mean(vals):.2f}')
    ax.set_title(nm, fontweight='bold'); ax.legend(fontsize=8); ax.grid(True, alpha=0.2)
plt.suptitle('Solo SSE/(n-p) es insesgado', fontweight='bold')
plt.tight_layout(); plt.show()

---
## 🟡 Ejercicio 2 — Detectar y cuantificar heterocedasticidad

1. Generar datos con y sin heterocedasticidad
2. Graficar residuos vs ajustados para ambos casos
3. Aplicar el test de Breusch-Pagan manual
4. Mostrar que OLS sigue siendo insesgado pero ya no es BLUE bajo heterocedasticidad

In [ ]:
np.random.seed(SEED)
n_h = 200
x_h = np.random.uniform(1, 20, n_h)
X_h = np.column_stack([np.ones(n_h), x_h])
beta_h = np.array([5.0, 2.0])

# Caso 1: homocedastico
y_homo = beta_h[0] + beta_h[1]*x_h + np.random.normal(0, 3, n_h)

# Caso 2: heterocedastico (σ crece con x)
sigma_h = 0.5 * x_h
y_hete  = beta_h[0] + beta_h[1]*x_h + np.random.normal(0, sigma_h)

# --- Tu código aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
n_h = 200; x_h = np.random.uniform(1, 20, n_h)
X_h = np.column_stack([np.ones(n_h), x_h])
beta_h = np.array([5., 2.])
y_homo = beta_h[0]+beta_h[1]*x_h+np.random.normal(0, 3, n_h)
y_hete = beta_h[0]+beta_h[1]*x_h+np.random.normal(0, 0.5*x_h)

def breusch_pagan(X, e):
    n_bp = len(e)
    e2   = e**2
    b_aux, *_ = np.linalg.lstsq(X, e2, rcond=None)
    e2h  = X @ b_aux
    R2   = 1 - ((e2 - e2h)**2).sum() / ((e2 - e2.mean())**2).sum()
    LM   = n_bp * R2
    p_v  = 1 - stats.chi2.cdf(LM, df=X.shape[1]-1)
    return LM, p_v

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (y_k, label) in zip(axes, [(y_homo,'Homocedastico'), (y_hete,'Heterocedastico')]):
    b_k, *_ = np.linalg.lstsq(X_h, y_k, rcond=None)
    e_k     = y_k - X_h @ b_k
    ax.scatter(X_h @ b_k, e_k, alpha=0.45, s=14, color='#2176AE', edgecolors='none')
    ax.axhline(0, color='#C0392B', lw=1.5, linestyle='--')
    LM_k, p_k = breusch_pagan(X_h, e_k)
    ax.set_title(f'{label}\nBP: LM={LM_k:.2f}, p={p_k:.4f} → hom.: {p_k > 0.05}',
                 fontweight='bold')
    ax.set_xlabel('ŷ'); ax.set_ylabel('Residuos'); ax.grid(True, alpha=0.2)
plt.tight_layout(); plt.show()

# Insesgadez bajo heterocedasticidad (OLS sigue siendo insesgado)
np.random.seed(SEED)
n_s2 = 3000
b_ols_h = np.zeros((n_s2, 2))
for s in range(n_s2):
    sig_s = 0.5 * x_h
    y_s   = beta_h[0]+beta_h[1]*x_h+np.random.normal(0, sig_s)
    b_ols_h[s], *_ = np.linalg.lstsq(X_h, y_s, rcond=None)
print('Bajo heterocedasticidad, OLS sigue siendo insesgado:')
print(f'  E[β̂₀] = {b_ols_h[:,0].mean():.4f}  (real = {beta_h[0]})')
print(f'  E[β̂₁] = {b_ols_h[:,1].mean():.4f}  (real = {beta_h[1]})')
print('Pero la varianza ya no es la mínima → ya no es BLUE → usar WLS')

---
## 🟡 Ejercicio 3 — Panel de diagnóstico completo

Dado el dataset de ventas abajo, construir el panel completo de 4 gráficos y hacer los 3 tests estadísticos. Concluir si los supuestos se cumplen.

In [ ]:
np.random.seed(7)
n_ej3 = 280
antiguedad_e = np.random.uniform(0, 20, n_ej3)
tamanio_e    = np.random.uniform(20, 200, n_ej3)
zona_e       = np.random.choice([0,1,2], n_ej3).astype(float)
ventas_e     = (200 + 15*antiguedad_e + 2.5*tamanio_e
                + 50*zona_e + np.random.normal(0, 80, n_ej3))

X_e  = np.column_stack([np.ones(n_ej3), StandardScaler().fit_transform(
           np.column_stack([antiguedad_e, tamanio_e, zona_e]))])

# --- Tu panel de diagnóstico aquí ---


In [ ]:
# ✅ SOLUCIÓN
np.random.seed(7)
n_ej3=280
ant_e=np.random.uniform(0,20,n_ej3); tam_e=np.random.uniform(20,200,n_ej3)
zon_e=np.random.choice([0,1,2],n_ej3).astype(float)
ven_e=200+15*ant_e+2.5*tam_e+50*zon_e+np.random.normal(0,80,n_ej3)
X_e=np.column_stack([np.ones(n_ej3),StandardScaler().fit_transform(
    np.column_stack([ant_e,tam_e,zon_e]))])
b_e,*_=np.linalg.lstsq(X_e,ven_e,rcond=None)
yh_e=X_e@b_e; e_e=ven_e-yh_e
n_e,p_e=X_e.shape; s2_e=(e_e@e_e)/(n_e-p_e); e_std_e=e_e/np.sqrt(s2_e)

fig,axes=plt.subplots(2,2,figsize=(13,9))
axes[0,0].scatter(yh_e,e_e,alpha=0.4,s=14,color='#2176AE',edgecolors='none')
axes[0,0].axhline(0,color='#C0392B',lw=1.5,linestyle='--')
axes[0,0].set(xlabel='ŷ',ylabel='Residuos',title='1. Residuos vs Ajustados')
axes[0,0].grid(True,alpha=0.2)
stats.probplot(e_std_e,dist='norm',plot=axes[0,1])
axes[0,1].set_title('2. Q-Q Plot',fontweight='bold')
axes[0,1].lines[0].set(color='#2176AE',alpha=0.5,markersize=4)
axes[0,1].lines[1].set(color='#C0392B',lw=2)
axes[1,0].scatter(yh_e,np.sqrt(np.abs(e_std_e)),alpha=0.4,s=14,
                  color='#5BA4CF',edgecolors='none')
axes[1,0].set(xlabel='ŷ',ylabel='√|e_std|',title='3. Scale-Location')
axes[1,0].grid(True,alpha=0.2)
axes[1,1].plot(e_e,'.',alpha=0.4,markersize=4,color='#2176AE')
axes[1,1].axhline(0,color='#C0392B',lw=1.5,linestyle='--')
axes[1,1].set(xlabel='Índice',ylabel='Residuos',title='4. Residuos vs Orden')
axes[1,1].grid(True,alpha=0.2)
plt.suptitle(f'Panel diagnóstico — Ventas (R²={r2_score(ven_e,yh_e):.3f})',
             fontweight='bold'); plt.tight_layout(); plt.show()

_,p_sw=stats.shapiro(e_e[:250])
dw=np.sum(np.diff(e_e)**2)/(e_e@e_e)
e2_e=e_e**2; b_bp,*_=np.linalg.lstsq(X_e,e2_e,rcond=None)
R2_bp=r2_score(e2_e, X_e@b_bp); LM_bp=n_e*R2_bp
p_bp=1-stats.chi2.cdf(LM_bp,df=p_e-1)
print('─'*50); print('TESTS DE SUPUESTOS')
print(f'  Shapiro-Wilk (normalidad):  p={p_sw:.4f}  OK: {p_sw>0.05}')
print(f'  Durbin-Watson (autocorr):   DW={dw:.4f}  OK: {1.5<dw<2.5}')
print(f'  Breusch-Pagan (homoced):   p={p_bp:.4f}  OK: {p_bp>0.05}')
print('─'*50)

---
## 🔴 Ejercicio 4 — Desafío: comparar OLS vs estimador alternativo bajo GM

Implementá una comparación entre OLS y dos estimadores alternativos bajo distintos escenarios de supuestos. Para cada escenario, reportar sesgo y varianza de β̂₁.

In [ ]:
def comparar_estimadores(n_obs, sigma_fn, n_sim=2000, escenario=''):
    """
    Compara OLS vs dos alternativas bajo distintos errores.
    sigma_fn(x): función que devuelve σ para cada observación x.
    """
    np.random.seed(SEED)
    beta_v = np.array([3.0, 2.0])
    x_v    = np.random.uniform(0, 10, n_obs)
    X_v    = np.column_stack([np.ones(n_obs), x_v])

    # --- Tu implementación aquí ---
    pass

# Tres escenarios
comparar_estimadores(100, lambda x: np.ones_like(x)*2,   escenario='Homocedastico GM4 OK')
comparar_estimadores(100, lambda x: 0.3*x,               escenario='Heterocedastico GM4 ✗')
comparar_estimadores(100, lambda x: np.ones_like(x)*2,   escenario='n pequeño n=30')


In [ ]:
# ✅ SOLUCIÓN
def comparar_estimadores(n_obs, sigma_fn, n_sim=2000, escenario=''):
    np.random.seed(SEED)
    beta_v=np.array([3.,2.])
    x_v=np.random.uniform(0,10,n_obs)
    X_v=np.column_stack([np.ones(n_obs),x_v])
    b_ols=np.zeros((n_sim,2))
    b_med=np.zeros((n_sim,2))  # alternativo: mediana ponderada
    b_sub=np.zeros((n_sim,2))  # alternativo: submuestra aleatoria 50%
    for s in range(n_sim):
        sig=sigma_fn(x_v)
        eps=np.random.normal(0,sig)
        y_s=X_v@beta_v+eps
        b_ols[s],*_=np.linalg.lstsq(X_v,y_s,rcond=None)
        # alternativo: usar solo obs con bajo error esperado
        idx=np.argsort(sig)[:n_obs//2]
        if len(idx)>=2:
            b_sub[s],*_=np.linalg.lstsq(X_v[idx],y_s[idx],rcond=None)
        # alternativo: añadir ruido al diseño
        X_noisy=X_v+np.random.randn(*X_v.shape)*0.1
        b_med[s],*_=np.linalg.lstsq(X_noisy,y_s,rcond=None)

    print(f'\n=== {escenario} ===')
    print(f'{"Estimador":20s} {"E[β̂₁]":>8s} {"Sesgo":>8s} {"Var(β̂₁)":>10s}')
    print('─'*50)
    for nm, bh in [('OLS',b_ols),('Sub-muestra',b_sub),('X+ruido',b_med)]:
        E  = bh[:,1].mean()
        bias = E - beta_v[1]
        var  = bh[:,1].var()
        print(f'  {nm:18s} {E:>8.4f} {bias:>8.4f} {var:>10.4f}')

comparar_estimadores(100, lambda x: np.ones_like(x)*2,   escenario='Homocedastico GM4 OK')
comparar_estimadores(100, lambda x: 0.3*x,               escenario='Heterocedastico GM4 X')
np.random.seed(SEED)
comparar_estimadores(30,  lambda x: np.ones_like(x)*2,   escenario='n pequeño n=30')

---
<div style="background:#0F3D6E;color:white;padding:20px 24px;border-radius:8px;">
<strong>Próxima clase — Lunes</strong><br>
R² y R² ajustado · Prueba F general · t-tests · Intervalos de confianza para β̂<br>
<em>Docente: Josef Rodriguez · Curso 8 · Modelos Estadísticos</em>
</div>